In [1]:
import numpy as np
import heapq
from nltk.translate.bleu_score import sentence_bleu
import re


def extract_formulas(response):
    # Define regular patterns for formulas
    patterns = [
        r'\\\[([^\]]*?)\\\]',     # \[ \]
        r'\\\(([^\)]*?)\\\)',     # \( \)
        r'\$([^\$]*?)\$'          # $ $
    ]
    
    formulas = set()
    
    for pattern in patterns:
        matches = re.findall(pattern, response)
        formulas.update(matches)
    return list(formulas) 


def calculate_unique_diversity(formulas, current_index):
    if not formulas[current_index]:  # 如果当前response为空
        return 0
    
    # 获取所有其他response中的公式
    other_formulas = set()
    for i in range(len(formulas)):
        if i != current_index:
            other_formulas.update(formulas[i])
    
    # 获取当前response中的公式
    current_formulas = set(formulas[current_index])
    
    # 计算只在当前response中出现的公式（独特公式）
    unique_formulas = current_formulas - other_formulas
    
    # 计算多样性指标 D_eq
    D_eq = len(unique_formulas) / len(formulas[current_index])

    # print(f'{len(unique_formulas)} / {len(formulas[current_index])}')
    
    return D_eq

def calculate_equation_matrix(group_rollouts):
    formulas = []
    for i in range(len(group_rollouts)):
        formulas.append(extract_formulas(group_rollouts[i]))
    
    diversity = []

    for i in range(len(formulas)):
        diversity.append(calculate_unique_diversity(formulas, i))
    return np.array(diversity)
    # return sum(diversity)/len(diversity)

def calculate_belu_matrix(group_rollouts):
    n = len(group_rollouts)
    similarity_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            # calculate BLEU score
            reference_i = [group_rollouts[i].split()]
            candidate_j = group_rollouts[j].split()
            bleu_i_j = sentence_bleu(reference_i, candidate_j)
            
            reference_j = [group_rollouts[j].split()]
            candidate_i = group_rollouts[i].split()
            bleu_j_i = sentence_bleu(reference_j, candidate_i)
            
            # Similarity is bidirectional
            similarity = (bleu_i_j + bleu_j_i) / 2
            similarity_matrix[i][j] = similarity
            similarity_matrix[j][i] = similarity

    # avg_similarities = np.sum(similarity_matrix, axis=1) / (n-1)
    return similarity_matrix
    # return avg_similarities.mean()

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from verl.utils.dataset.rl_dataset import RLHFDataset, collate_fn
from torch.utils.data import DataLoader

In [3]:
model_path = "/mnt/petrelfs/huzican/R1/rlvr_div/checkpoints/div/rs_0.01_div+rule_equ_piecewise/actor/global_step_100"
tokenizer = AutoTokenizer.from_pretrained(model_path)
policy_model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype="auto", device_map="auto")
# policy_model = AutoModelForCausalLM.from_pretrained(model_path).to('cuda:7')

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

In [4]:
train_data_path = "dataset/train.parquet"
train_dataset = RLHFDataset(parquet_files=train_data_path,
                            tokenizer=tokenizer,
                            prompt_key='prompt',
                            max_prompt_length=1024,
                            filter_prompts=True,
                            return_raw_chat=False,
                            truncation='error')

train_dataloader = DataLoader(dataset=train_dataset,
                            batch_size=1,
                            shuffle=True,
                            drop_last=True,
                            collate_fn=collate_fn)

original dataset len: 8523
filter dataset len: 8512


In [5]:
test_data=train_dataset[2]
seq = tokenizer.batch_decode([test_data['input_ids']],skip_special_tokens=True)
# 准备输入
input_ids = test_data['input_ids'].to(policy_model.device).view(1,-1)
attention_mask = test_data['attention_mask'].to(policy_model.device).view(1,-1)
group_rollout = []
group_hidden_state= []
for i in range(8):
    # 生成文本
    with torch.no_grad():
        gene = policy_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=8192,  # 生成新token的数量
            do_sample=True,
            temperature=1.2,
            top_p=1.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            output_hidden_states=True,
            return_dict_in_generate=True,
            # repetition_penalty=1.1  # 避免重复
        )

    group_hidden_state.append(gene['hidden_states'][-1][-1].squeeze(0).squeeze(0))
    
    # # 解码生成的序列
    original_length = input_ids.shape[1]
    new_tokens = gene['sequences'][:, original_length:]
    generated_texts = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    group_rollout.append(generated_texts)

for i in range(len(group_rollout)):
    print(f"********{i}**********")
    print(group_rollout[i])
    print("******************")



From v4.47 onwards, when a model cache is to be returned, `generate` will return a `Cache` instance instead by default (as opposed to the legacy tuple of tuples format). If you want to keep returning the legacy format, please set `return_legacy_cache=True`.


KeyboardInterrupt: 

In [6]:
rollout_list = []
for response in group_rollout:
    rollout_list.append(response[0])
belu = calculate_belu_matrix(rollout_list)
equ = calculate_equation_matrix(rollout_list)

/mnt/petrelfs/huzican/anaconda3/envs/llm/lib/python3.9/site-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/mnt/petrelfs/huzican/anaconda3/envs/llm/lib/python3.9/site-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


In [7]:
import os
policy_model.to('cpu')
model_path = "/mnt/petrelfs/huzican/R1/rlvr_div/checkpoints/div/cl_hidden_1_5_coeff_0_1_clip_020_028_solve_none/actor/global_step_100"
cl_tokenizer = AutoTokenizer.from_pretrained(model_path)
cl_model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype="auto", device_map="auto")

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

In [8]:
n=len(group_rollout)
cl_hidden_state = []
for i in range(n):
    with torch.no_grad():
        gene_ids = cl_tokenizer(group_rollout[i],padding=True,truncation=True,return_tensors='pt').to(cl_model.device)
        output=cl_model(input_ids=torch.cat([input_ids,gene_ids['input_ids']],dim=1),
                 attention_mask=torch.cat([attention_mask,gene_ids['attention_mask']],dim=1),
                 output_hidden_states=True)
        cl_hidden_state.append(output.hidden_states[-1][-1][-1].squeeze(0).squeeze(0))

In [9]:
model_path = "/mnt/petrelfs/share_data/zhangshilin/baseline_clp_02_028/actor/global_step_100"
baseline_tokenizer = AutoTokenizer.from_pretrained(model_path)
baseline_model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype="auto", device_map="auto")

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

In [10]:

baseline_hidden_state = []
for i in range(n):
    with torch.no_grad():
        gene_ids = baseline_tokenizer(group_rollout[i],padding=True,truncation=True,return_tensors='pt').to(baseline_model.device)
        output=baseline_model(input_ids=torch.cat([input_ids,gene_ids['input_ids']],dim=1),
                 attention_mask=torch.cat([attention_mask,gene_ids['attention_mask']],dim=1),
                 output_hidden_states=True)
        baseline_hidden_state.append(output.hidden_states[-1][-1][-1].squeeze(0).squeeze(0))

In [11]:
from torch import nn
cl_hidden_state_list,hidden_state_list,baseline_hidden_state_list = [],[],[]
for i in range(n):
    hidden_states_norm = nn.functional.normalize(group_hidden_state[i], p=2,dim=0)
    cl_hidden_states_norm = nn.functional.normalize(cl_hidden_state[i], p=2,dim=0)
    baseline_hidden_states_norm = nn.functional.normalize(baseline_hidden_state[i], p=2,dim=0)
    cl_hidden_state_list.append(cl_hidden_states_norm)
    hidden_state_list.append(hidden_states_norm)
    baseline_hidden_state_list.append(baseline_hidden_states_norm)
# for hidden_state in cl_hidden_state:
#     # print(hidden_state.shape)
#     hidden_states_norm = nn.functional.normalize(hidden_state, p=2,dim=0)
#     cl_hidden_state_list.append(hidden_states_norm)
# n = len(hidden_state_list)
similar_logits = np.zeros((n, n))
cl_similar_logits = np.zeros((n, n))
baseline_similar_logits = np.zeros((n, n))
for i in range(n):
    for j in range(i+1, n):
        logits_i_j = torch.mm(hidden_state_list[i].reshape(1,-1), hidden_state_list[j].reshape(-1,1))
        # logits_j_i = torch.mm(hidden_state_list[j].reshape(1,-1), hidden_state_list[i].reshape(-1,1))
        cl_logits_i_j = torch.mm(cl_hidden_state_list[j].reshape(1,-1), cl_hidden_state_list[i].reshape(-1,1))
        baseline_logits_i_j = torch.mm(baseline_hidden_state_list[j].reshape(1,-1), baseline_hidden_state_list[i].reshape(-1,1))
        # print(logits_i_j, logits_j_i)
        
        # Similarity is bidirectional
        # similarity = (logits_i_j + logits_j_i) / 2
        similar_logits[i][j] = logits_i_j
        similar_logits[j][i] = logits_i_j
        baseline_similar_logits[i][j] = baseline_logits_i_j
        baseline_similar_logits[j][i] = baseline_logits_i_j
        cl_similar_logits[i][j] = cl_logits_i_j
        cl_similar_logits[j][i] = cl_logits_i_j
    # print(np.argsort(similar_logits[i]).tolist())



In [12]:
for i in range(n):
    print("belu:", np.argsort(belu[i])[1:])
    print("start_hidden:",np.argsort(similar_logits[i])[1:])
    print("cl_hidden:",np.argsort(cl_similar_logits[i])[1:])
    print("baseline_hidden:",np.argsort(baseline_similar_logits[i])[1:])
    print("*********************")
    print("belu:", np.mean(belu[i]))
    print("start_hidden:",np.mean(similar_logits[i]))
    print("cl_hidden:",np.mean(cl_similar_logits[i]))
    print("baseline_hidden:",np.mean(baseline_similar_logits[i]))
    print("_____________________")
print("cl:",cl_similar_logits)
print("baseline:",baseline_similar_logits)

belu: [5 1 4 3 2 6 7]
start_hidden: [2 7 4 5 1 3 6]
cl_hidden: [7 2 4 1 5 3 6]
baseline_hidden: [7 2 5 4 1 6 3]
*********************
belu: 0.09044397978181065
start_hidden: 0.6484375
cl_hidden: 0.4749155268073082
baseline_hidden: 0.644912838935852
_____________________
belu: [4 5 3 2 7 0 6]
start_hidden: [7 2 0 5 3 4 6]
cl_hidden: [4 3 7 2 0 6 5]
baseline_hidden: [7 3 0 5 2 4 6]
*********************
belu: 0.03239776285106584
start_hidden: 0.71044921875
cl_hidden: 0.4577162601053715
baseline_hidden: 0.6943164840340614
_____________________
belu: [5 1 6 4 0 3 7]
start_hidden: [5 0 7 3 1 4 6]
cl_hidden: [7 5 0 3 4 1 6]
baseline_hidden: [7 3 0 5 4 6 1]
*********************
belu: 0.07728717480855692
start_hidden: 0.61083984375
cl_hidden: 0.4108048267662525
baseline_hidden: 0.6019608601927757
_____________________
belu: [1 5 6 4 7 0 2]
start_hidden: [7 5 2 1 0 6 4]
cl_hidden: [7 1 5 2 6 0 4]
baseline_hidden: [7 2 5 1 6 0 4]
*********************
belu: 0.05696972371203379
start_hidden: 0.6